# Tech Challenge — Classificação da Qualidade de Vinhos
**POSTECH | Data Analytics  — Fase 2**

## Objetivo
Construir e comparar modelos de classificação para prever se um vinho é de **alta qualidade** (`quality ≥ 7`) ou de **baixa/média qualidade** (`quality < 7`), usando as propriedades físico-químicas disponíveis no arquivo `WineQT.csv`.

## 1. Preparação do ambiente e carregamento dos dados

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    confusion_matrix, RocCurveDisplay
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

csv_path = "/content/WineQT.csv"
if not os.path.exists(csv_path):
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    except ImportError:
        csv_path = "WineQT.csv"

if csv_path is None or not os.path.exists(csv_path):
    raise FileNotFoundError("Não encontrei WineQT.csv. Envie o arquivo para o Colab ou coloque-o no diretório de trabalho.")

df = pd.read_csv(csv_path)
print(f"Dimensões da base: {df.shape[0]} linhas × {df.shape[1]} colunas")
display(df.head())


## 2. Compreensão do problema
A variável-alvo original é `quality`, uma nota ordinal atribuída ao vinho. Para transformar o problema em classificação binária, aplicamos a regra do enunciado:

- **Alta qualidade (1):** nota `quality ≥ 7`
- **Baixa/média qualidade (0):** nota `quality < 7`

A coluna `Id`, quando presente, é apenas identificadora e não deve ser usada como característica preditiva. Ela será removida dos atributos.

In [ ]:
if "quality" not in df.columns:
    raise ValueError("A base precisa conter a coluna 'quality'.")

print("Tipos de dados:")
display(df.dtypes.to_frame("tipo"))
print("\nResumo estatístico:")
display(df.describe().T)

df = df.copy()
df["high_quality"] = (df["quality"] >= 7).astype(int)

print("\nDistribuição da nota original:")
display(df["quality"].value_counts().sort_index().rename("quantidade").to_frame())
print("\nDistribuição do alvo binário:")
display(df["high_quality"].value_counts().rename(index={0: "Baixa/média (0)", 1: "Alta (1)"}).to_frame("quantidade"))
print("\nProporção do alvo binário (%):")
display((df["high_quality"].value_counts(normalize=True) * 100).rename(index={0: "Baixa/média (0)", 1: "Alta (1)"}).round(2).to_frame("percentual"))


## 3. Análise exploratória de dados (EDA)
Vamos verificar qualidade dos dados, distribuição das variáveis, equilíbrio entre classes, correlações e possíveis valores extremos. Outliers serão investigados, não removidos automaticamente: em dados físico-químicos, valores extremos podem ser observações reais.

In [ ]:
# Valores ausentes, duplicatas e estrutura
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("ausentes"))
print(f"Linhas duplicadas: {df.duplicated().sum()}")
print(f"Linhas totalmente duplicadas: {df.drop(columns=['high_quality']).duplicated().sum()}")

# Balanceamento das classes
class_counts = df["high_quality"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
sns.barplot(x=["Baixa/média (<7)", "Alta (≥7)"], y=[class_counts.get(0, 0), class_counts.get(1, 0)])
plt.title("Balanceamento da variável-alvo")
plt.ylabel("Número de amostras")
plt.xlabel("")
plt.show()

# Histogramas das variáveis numéricas
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
plot_cols = [c for c in numeric_cols if c not in ["Id", "high_quality"]]
df[plot_cols].hist(figsize=(16, 12), bins=25, edgecolor="black")
plt.suptitle("Distribuição das variáveis numéricas", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Matriz de correlação: útil para relações lineares, mas não implica causalidade
corr_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in ["Id"]]
plt.figure(figsize=(12, 9))
sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", square=False)
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

target_corr = df[plot_cols + ["high_quality"]].corr(numeric_only=True)["high_quality"].drop("high_quality").sort_values()
print("Correlação linear de cada variável com a classe binária:")
display(target_corr.to_frame("correlação com high_quality"))


In [ ]:
# Inspeção de possíveis outliers pelo critério IQR
# O critério é exploratório: não significa que os registros devam ser excluídos.
outlier_rows = []
for col in plot_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({
        "variável": col, "limite inferior IQR": lower,
        "limite superior IQR": upper, "nº de possíveis outliers": int(n_out),
        "% da base": round(100 * n_out / len(df), 2)
    })
outlier_summary = pd.DataFrame(outlier_rows).sort_values("nº de possíveis outliers", ascending=False)
display(outlier_summary)

plt.figure(figsize=(15, 7))
sns.boxplot(data=df[plot_cols], orient="h")
plt.title("Boxplots para inspeção de valores extremos")
plt.tight_layout()
plt.show()


## 4. Pré-processamento e divisão treino/teste
Separaremos 20% dos dados para teste e usaremos estratificação para preservar aproximadamente a proporção das classes. A coluna `quality` original e a coluna `Id` não entrarão como preditoras: a primeira define diretamente o alvo e a segunda é identificadora.

Os dois modelos serão:
1. **Regressão Logística** — modelo linear de referência; exige padronização das variáveis.
2. **Random Forest** — conjunto de árvores capaz de capturar relações não lineares e interações.

A imputação e a padronização ficam dentro de `Pipeline`, para que sejam ajustadas somente com os dados de treino e reduzir o risco de vazamento de dados.

In [ ]:
target = "high_quality"
excluded = ["quality", "high_quality", "Id"]
feature_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in excluded]

X = df[feature_cols].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Variáveis preditoras ({len(feature_cols)}): {feature_cols}")
print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print("\nProporção das classes — treino:")
display(y_train.value_counts(normalize=True).sort_index().rename(index={0:"Baixa/média",1:"Alta"}).mul(100).round(2).to_frame("%"))
print("\nProporção das classes — teste:")
display(y_test.value_counts(normalize=True).sort_index().rename(index={0:"Baixa/média",1:"Alta"}).mul(100).round(2).to_frame("%"))

models = {
    "Regressão Logística": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}


## 5. Treinamento e avaliação dos modelos
A comparação principal será feita no conjunto de teste, que fica separado durante o treinamento. Como a classe de alta qualidade pode ser minoritária, o **F1-score** e o **recall da classe positiva** são especialmente informativos, junto com precisão e ROC-AUC.

In [ ]:
results = []
fitted_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    fitted_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "Modelo": name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão (alta qualidade)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (alta qualidade)": recall_score(y_test, y_pred, zero_division=0),
        "F1-score (alta qualidade)": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Modelo").sort_values("F1-score (alta qualidade)", ascending=False)
display(results_df.style.format("{:.3f}"))

for name in fitted_models:
    print(f"\n===== {name} =====")
    print(classification_report(
        y_test, predictions[name],
        labels=[0, 1], target_names=["Baixa/média qualidade", "Alta qualidade"],
        zero_division=0
    ))


In [ ]:
# Matrizes de confusão
fig, axes = plt.subplots(1, len(fitted_models), figsize=(6 * len(fitted_models), 4))
if len(fitted_models) == 1:
    axes = [axes]
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["Baixa/média", "Alta"],
        cmap="Blues", values_format="d", ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

# Curvas ROC
plt.figure(figsize=(7, 6))
for name, y_prob in probabilities.items():
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Referência aleatória")
plt.title("Curvas ROC — conjunto de teste")
plt.tight_layout()
plt.show()


## 6. Interpretação: quais variáveis mais influenciam as previsões?
Na Random Forest, a importância mede quanto cada variável contribui para reduzir impureza nas árvores. Na Regressão Logística, os coeficientes padronizados indicam a direção da associação dentro do modelo. Essas medidas são explicativas do comportamento do modelo, não evidência de causalidade.

In [ ]:
# Importância de variáveis da Random Forest
rf = fitted_models["Random Forest"].named_steps["model"]
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(rf_importance.rename("importância").to_frame())

plt.figure(figsize=(8, 5))
sns.barplot(x=rf_importance.head(10).values, y=rf_importance.head(10).index)
plt.title("Top 10 variáveis — importância da Random Forest")
plt.xlabel("Importância")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Coeficientes da Regressão Logística
lr = fitted_models["Regressão Logística"].named_steps["model"]
lr_coef = pd.Series(lr.coef_[0], index=feature_cols).sort_values()
print("Coeficientes da Regressão Logística (valores positivos favorecem a classe alta no modelo):")
display(lr_coef.to_frame("coeficiente").style.format("{:.3f}"))

plt.figure(figsize=(8, 5))
lr_coef.plot(kind="barh")
plt.axvline(0, linestyle="--")
plt.title("Coeficientes da Regressão Logística")
plt.xlabel("Coeficiente (atributos padronizados)")
plt.tight_layout()
plt.show()


## 7. Conclusões executivas e implicações para a produção
A célula abaixo gera uma síntese automática com base nos resultados efetivamente obtidos ao executar o notebook. Revise o texto final antes de apresentar: ele deve refletir as métricas, o objetivo operacional e os limites da análise.

In [ ]:
best_model_name = results_df.index[0]
best_row = results_df.iloc[0]
top_features = rf_importance.head(3).index.tolist()

print("SÍNTESE EXECUTIVA")
print("-" * 60)
print(f"Modelo com maior F1-score no teste: {best_model_name}.")
print(f"F1-score: {best_row['F1-score (alta qualidade)']:.3f} | "
      f"Recall: {best_row['Recall (alta qualidade)']:.3f} | "
      f"Precisão: {best_row['Precisão (alta qualidade)']:.3f} | "
      f"ROC-AUC: {best_row['ROC-AUC']:.3f}.")
print(f"Entre as variáveis mais importantes na Random Forest estão: {', '.join(top_features)}.")
print("\nImplicações práticas:")
print("1. Usar o modelo como apoio à triagem e ao controle de qualidade, não como decisão definitiva.")
print("2. Investigar as variáveis mais relevantes junto a enólogos e especialistas de processo.")
print("3. Validar o modelo com novas safras, regiões e dados de produção antes de uso operacional.")
print("4. Ajustar o limiar de classificação conforme o custo de falsos positivos e falsos negativos.")
print("\nLimitações:")
print("- A base é observacional e pode não representar todos os vinhos ou condições produtivas.")
print("- A nota foi binarizada em 7, descartando diferenças entre notas dentro de cada grupo.")
print("- Importância e correlação não demonstram causalidade.")


## 8. Exportação dos resultados
Salva a tabela de comparação e a importância das variáveis em CSV, facilitando a inclusão no repositório GitHub e na apresentação executiva.

In [ ]:
results_df.to_csv("comparacao_modelos.csv", encoding="utf-8-sig")
rf_importance.rename("importancia").to_csv("importancia_variaveis.csv", encoding="utf-8-sig")
print("Arquivos gerados: comparacao_modelos.csv e importancia_variaveis.csv")

# No Google Colab, descomente as linhas abaixo para baixar os arquivos:
# from google.colab import files
# files.download("comparacao_modelos.csv")
# files.download("importancia_variaveis.csv")
